In [ ]:
import os
# Root of the local Bedretto dataset (not distributed with this repo, see README)
DATA_DIR = os.environ.get("BEDRETTO_DATA_DIR", "data") + "/"
os.chdir(DATA_DIR)  # notebook uses paths relative to the dataset root

In [ ]:
import numpy as np
import h5py
import os
from scipy import signal
from obspy import Trace
import pandas as pd
import glob
import matplotlib.pyplot as plt
import random
import scipy.signal
from sklearn.preprocessing import StandardScaler, RobustScaler

In [ ]:
filename = "waveforms.hdf5"
hdf5_file = h5py.File(filename, "r")

In [ ]:
def save_hdf5_to_npz(hdf5_filename, output_dir="npz_files"):
    """
    Save HDF5 waveform data to individual .npz files for each bucket
    
    Parameters:
    - hdf5_filename: Path to the HDF5 file
    - output_dir: Directory to save .npz files
    """
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    with h5py.File(hdf5_filename, 'r') as f:
        print("Saving buckets to .npz files...")
        
        # Save each bucket as a separate .npz file
        for bucket_num in range(13):  # buckets 0-16
            bucket_key = f'data/bucket{bucket_num}'
            
            if bucket_key in f:
                bucket_data = f[bucket_key][...]
                num_traces = bucket_data.shape[0]
                
                # Save bucket data
                output_path = os.path.join(output_dir, f'bucket_{bucket_num:02d}.npz')
                np.savez_compressed(output_path, 
                                  waveforms=bucket_data,
                                  bucket_number=bucket_num,
                                  num_traces=num_traces,
                                  original_sampling_rate=200000)
                
                print(f"Saved bucket {bucket_num}: {num_traces} traces -> {output_path}")
            else:
                print(f"Warning: {bucket_key} not found in HDF5 file")
        
        # Also save data format information if available
        try:
            format_info = {}
            format_info['component_order'] = f['data_format/component_order'][()]
            format_info['dimension_order'] = f['data_format/dimension_order'][()]
            format_info['instrument_response'] = f['data_format/instrument_response'][()]
            format_info['measurement'] = f['data_format/measurement'][()]
            format_info['unit'] = f['data_format/unit'][()]
            
            format_path = os.path.join(output_dir, 'data_format.npz')
            np.savez(format_path, **format_info)
            print(f"Saved data format info -> {format_path}")
            
        except KeyError:
            print("Warning: Could not save data format information")
    
    print(f"\nAll files saved to: {os.path.abspath(output_dir)}")
    return output_dir

# Test function to verify saved data
def verify_npz_files(npz_dir="npz_files"):
    """
    Verify that .npz files were saved correctly
    """
    print("Verifying saved .npz files...")
    
    total_traces = 0
    for bucket_num in range(13):
        npz_path = os.path.join(npz_dir, f'bucket_{bucket_num:02d}.npz')
        
        if os.path.exists(npz_path):
            data = np.load(npz_path)
            waveforms = data['waveforms']
            num_traces = data['num_traces']
            
            print(f"Bucket {bucket_num}: Shape {waveforms.shape}, {num_traces} traces")
            total_traces += num_traces
        else:
            print(f"Warning: {npz_path} not found")
    
    print(f"\nTotal traces across all buckets: {total_traces}")
    
    # Check data format file
    format_path = os.path.join(npz_dir, 'data_format.npz')
    if os.path.exists(format_path):
        format_data = np.load(format_path)
        print(f"\nData format info available: {list(format_data.keys())}")
    
    return total_traces

# Execute the saving process
if __name__ == "__main__":
    # Save the HDF5 data to .npz files
    output_directory = save_hdf5_to_npz("waveforms.hdf5")
    
    # Verify the saved files
    verify_npz_files(output_directory)

In [ ]:
def resample_npz_files(input_dir="npz_files", output_dir="npz_files_100kHz", 
                       original_rate=200000, target_rate=100000):
    """
    Resample .npz files from 200 kHz to 100 kHz 
    Params:
    - input_dir: Directory containing original .npz files
    - output_dir: Directory to save resampled .npz files
    - original_rate: Original sampling rate (Hz)
    - target_rate: Target sampling rate (Hz)
    """
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Resampling from {original_rate} Hz to {target_rate} Hz ")
    
    for bucket_num in range(13):
        input_path = os.path.join(input_dir, f'bucket_{bucket_num:02d}.npz')
        
        if os.path.exists(input_path):
            print(f"Processing bucket {bucket_num}...")
            
            # Load original data
            data = np.load(input_path)
            original_waveforms = data['waveforms']
            num_traces = data['num_traces']
            
            # Get dimensions
            n_traces, n_components, n_samples = original_waveforms.shape
            
            # Calculate new number of samples after resampling
            new_n_samples = int(n_samples * target_rate / original_rate)
            
            # Initialize resampled array
            resampled_waveforms = np.zeros((n_traces, n_components, new_n_samples))
            
            # Resample each trace
            for trace_idx in range(n_traces):
                for comp_idx in range(n_components):
                    # Create ObsPy Trace
                    tr = Trace(data=original_waveforms[trace_idx, comp_idx, :])
                    tr.stats.sampling_rate = original_rate
                    
                    # Resample using ObsPy
                    tr.resample(target_rate, no_filter=False) #should be false
                    
                    # Store resampled data
                    resampled_waveforms[trace_idx, comp_idx, :] = tr.data
                
                # Progress indicator
                if (trace_idx + 1) % 100 == 0:
                    print(f"  Processed {trace_idx + 1}/{n_traces} traces")
            
            # Save resampled data
            output_path = os.path.join(output_dir, f'bucket_{bucket_num:02d}_resampled.npz')
            np.savez_compressed(output_path,
                              waveforms=resampled_waveforms,
                              bucket_number=bucket_num,
                              num_traces=num_traces,
                              original_sampling_rate=original_rate,
                              resampled_sampling_rate=target_rate)
            
            print(f"  Saved bucket {bucket_num}: {original_waveforms.shape} → {resampled_waveforms.shape}")
            print(f"  File: {output_path}")
        
        else:
            print(f"Warning: {input_path} not found")
    
    # Copy data format file if it exists
    format_input = os.path.join(input_dir, 'data_format.npz')
    if os.path.exists(format_input):
        format_output = os.path.join(output_dir, 'data_format.npz')
        import shutil
        shutil.copy2(format_input, format_output)
        print(f"Copied data format file to {format_output}")
    
    print(f"\nResampling complete! Files saved to: {os.path.abspath(output_dir)}")
    return output_dir

# Execute resampling
output_dir = resample_npz_files()

In [ ]:
from scipy.signal import butter, filtfilt
def highpass_npz_files(input_dir="npz_files_100kHz", output_dir="npz_files_100kHz_highpass_bworder4_zerophase", sampling_rate=100000, cutoff_freq=200): 
    """
    Apply high-pass filter to remove low frequencies below 200 Hz
    Params:
    - input_dir: Directory containing original .npz files
    - output_dir: Directory to save filtered .npz files
    - sampling_rate: sampling rate (Hz)
    - cutoff_freq: cutoff frequency for high-pass filter (Hz)
    """
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Applying high-pass filter: removing frequencies below {cutoff_freq} Hz")
    order = 4
    nyq = 0.5 * sampling_rate              # Nyquist Frequency
    normal_cutoff = cutoff_freq / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    
    for bucket_num in range(13): 
        input_path = os.path.join(input_dir, f'bucket_{bucket_num:02d}_resampled.npz')
        
        if os.path.exists(input_path):
            print(f"Processing bucket {bucket_num}...")
            
            # Load original data
            data = np.load(input_path)
            original_waveforms = data['waveforms']
            num_traces = data['num_traces']
            
            # Get dimensions
            n_traces, n_components, n_samples = original_waveforms.shape

            # Apply high-pass filter
            filtered_waveforms = np.zeros_like(original_waveforms)
            for trace_idx in range(n_traces):
                for comp_idx in range(n_components):
                    # Apply highpass filter
                    filtered_waveforms[trace_idx, comp_idx, :] = filtfilt(b, a, original_waveforms[trace_idx, comp_idx, :])

                # Progress indicator
                if (trace_idx + 1) % 100 == 0:
                    print(f"  Processed {trace_idx + 1}/{n_traces} traces")
            
            # Save filtered data
            output_path = os.path.join(output_dir, f'bucket_{bucket_num:02d}_highpass.npz')
            np.savez_compressed(output_path,
                              waveforms=filtered_waveforms,
                              bucket_number=bucket_num,
                              num_traces=num_traces,
                              sampling_rate=sampling_rate,
                              highpass_freq=cutoff_freq)
            print(f"  Saved bucket {bucket_num}: {original_waveforms.shape} → {filtered_waveforms.shape}")
            print(f"  File: {output_path}")
        
        else:
            print(f"Warning: {input_path} not found")
    
    # Copy data format file if it exists
    format_input = os.path.join(input_dir, 'data_format.npz')
    if os.path.exists(format_input):
        format_output = os.path.join(output_dir, 'data_format.npz')
        import shutil
        shutil.copy2(format_input, format_output)
        print(f"Copied data format file to {format_output}")
    
    print(f"\nHigh-pass filtering complete! Files saved to: {os.path.abspath(output_dir)}")
    return output_dir

# Execute high-pass filtering
output_dir = highpass_npz_files()

In [ ]:
metadata = pd.read_csv('metadata.csv')

In [ ]:
def analyze_bucket_traces_time_domain(bucket_dir, metadata, num_buckets=13, num_samples=None, snr_threshold=3):
    """
    Analyze traces from bucket .npz files in time domain - PLOTS ONLY
    
    Parameters:
    - bucket_dir: Directory containing bucket .npz files (e.g., 'npz_files_100kHz_highpass_bworder4_zerophase')
    - metadata: Metadata DataFrame with trace information including SNR
    - num_buckets: Number of buckets to process (default 5)
    - num_samples: Number of traces to analyze per bucket (None = all traces)
    - snr_threshold: Minimum SNR threshold to include traces (default 3)
    """
    import matplotlib.pyplot as plt
    
    print(f"Analyzing traces from {num_buckets} buckets in {bucket_dir}...")
    
    # Load all traces from bucket files
    all_traces = []
    trace_lengths = []
    
    for bucket_num in range(num_buckets):
        bucket_path = os.path.join(bucket_dir, f'bucket_{bucket_num:02d}_highpass.npz')
        
        if os.path.exists(bucket_path):
            print(f"Processing bucket {bucket_num}...")
            
            # Load bucket data
            data = np.load(bucket_path)
            bucket_waveforms = data['waveforms']
            sampling_rate = data.get('sampling_rate', 100000)
            num_traces_in_bucket = bucket_waveforms.shape[0]
            
            # Determine how many traces to process from this bucket
            traces_to_process = num_traces_in_bucket
            if num_samples is not None:
                traces_to_process = min(num_samples, num_traces_in_bucket)
            
            # Process each trace in the bucket
            for trace_idx in range(traces_to_process):
                # Use the same comprehensive search as buckets_to_event_noise_files
                found = False
                for x in range(29990, 30011):
                    trace_name_candidate = f'bucket{bucket_num}${trace_idx},:1,:{x}'
                    mask = metadata['trace_name'] == trace_name_candidate
                    if mask.any():
                        trace_snr = metadata.loc[mask, 'trace_snr'].values[0]
                        found = True
                        break
                
                if not found:
                    continue
                
                # Skip traces below SNR threshold
                if trace_snr < snr_threshold:
                    continue
                
                # Extract trace data (single component, all samples)
                trace_data = bucket_waveforms[trace_idx, 0, :]  # [trace, component=0, samples]
                
                # Store trace and its length
                all_traces.append(trace_data)
                trace_lengths.append(len(trace_data))
            
            print(f"  Processed {traces_to_process} traces from bucket {bucket_num}")
        else:
            print(f"Warning: {bucket_path} not found")
    
    # Check if any traces were found
    if len(all_traces) == 0:
        print("No traces found meeting the criteria!")
        print(f"Checked {num_buckets} buckets with SNR threshold ≥ {snr_threshold}")
        return None
    
    # Check if all traces have the same length
    unique_lengths = set(trace_lengths)
    print(f"\nFound trace lengths: {unique_lengths}")
    
    # Instead of filtering, keep all traces regardless of their lengths
    # Pad shorter traces with NaN so all have the same length for statistics

    max_length = max(trace_lengths)
    print(f"Padding all traces to length: {max_length} samples")

    padded_traces = []
    for trace in all_traces:
        if len(trace) < max_length:
            # Pad with NaN at the end
            padded = np.pad(trace, (0, max_length - len(trace)), mode='constant', constant_values=np.nan)
            padded_traces.append(padded)
        else:
            padded_traces.append(trace)

    all_traces = padded_traces
    print(f"Kept {len(all_traces)} traces (padded to {max_length} samples)")
    
    # Convert to numpy array - now all traces should have the same length
    all_traces = np.array(all_traces)  # Shape: (n_traces, n_samples)
    
    print(f"\nLoaded {all_traces.shape[0]} traces, each with {all_traces.shape[1]} samples")
    print(f"SNR threshold: {snr_threshold}")
    print(f"Sampling rate: {sampling_rate} Hz")
    
    # Calculate statistics for plotting
    median_trace = np.median(all_traces, axis=0)
    mean_trace = np.mean(all_traces, axis=0)
    std_trace = np.std(all_traces, axis=0)
    percentile_25 = np.percentile(all_traces, 25, axis=0)
    percentile_75 = np.percentile(all_traces, 75, axis=0)
    percentile_5 = np.percentile(all_traces, 5, axis=0)
    percentile_95 = np.percentile(all_traces, 95, axis=0)
    
    # Create time axis (full trace duration)
    trace_duration = all_traces.shape[1] / sampling_rate
    time_axis = np.linspace(0, trace_duration, all_traces.shape[1])
    
    # Create time domain plot
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 12))
    
    # Plot 1: Median trace with percentile bands
    ax1.fill_between(time_axis, percentile_5, percentile_95, alpha=0.2, color='green', 
                    label='90th percentile band (5%-95%)')
    ax1.fill_between(time_axis, percentile_25, percentile_75, alpha=0.4, color='green', 
                    label='50th percentile band (25%-75%)')
    ax1.plot(time_axis, median_trace, 'g-', linewidth=2, label='Median trace')
    ax1.plot(time_axis, mean_trace, 'r--', linewidth=1, label='Mean trace')
    
    # Add vertical lines for noise and event sections
    ax1.axvline(x=0.005, color='blue', linestyle='--', linewidth=2, alpha=0.7, label='Noise section start')
    ax1.axvline(x=0.07, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Noise end / Event start')
    ax1.axvline(x=0.15, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Event section end')
    
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Amplitude')
    ax1.set_title(f'Bucket Traces Time Domain Analysis (train dataset, {len(all_traces)} traces, SNR ≥ {snr_threshold})')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Plot 2: Standard deviation over time
    ax2.plot(time_axis, std_trace, 'purple', linewidth=2, label='Standard deviation')
    
    # Add vertical lines for noise and event sections
    ax2.axvline(x=0.005, color='blue', linestyle='--', linewidth=2, alpha=0.7, label='Noise section start')
    ax2.axvline(x=0.07, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Noise end / Event start')
    ax2.axvline(x=0.15, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Event section end')
    
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Standard Deviation')
    ax2.set_title('Variability of Traces Over Time')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    # Plot 3: Example individual traces vs median
    ax3.plot(time_axis, all_traces[0], 'gray', alpha=0.5, label='Individual traces')
    for i in range(1, min(10, len(all_traces))):
        ax3.plot(time_axis, all_traces[i], 'gray', alpha=0.5)
    ax3.plot(time_axis, median_trace, 'g-', linewidth=2, label='Median trace')
    
    # Add vertical lines for noise and event sections
    ax3.axvline(x=0.005, color='blue', linestyle='--', linewidth=2, alpha=0.7, label='Noise section start')
    ax3.axvline(x=0.07, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Noise end / Event start')
    ax3.axvline(x=0.15, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Event section end')
    
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('Amplitude')
    ax3.set_title('Individual Traces vs Median (showing first 10 traces)')
    ax3.grid(True, alpha=0.3)
    ax3.legend()
    
    plt.tight_layout()
    plt.show()
    
    return {
        'median_trace': median_trace,
        'mean_trace': mean_trace,
        'std_trace': std_trace,
        'time_axis': time_axis,
        'sampling_rate': sampling_rate
    }

In [ ]:
results = analyze_bucket_traces_time_domain('npz_files_100kHz_highpass_bworder4_zerophase', metadata, snr_threshold=3)

In [ ]:
def buckets_to_event_noise_files(buckets, data_path_input, data_path_output_event, data_path_output_noise, metadata, sampling_rate=100000, snr_threshold=3):
    """
    Convert bucket files to separate event and noise files (single component) with SNR filtering
    
    Parameters:
    - buckets: list of bucket numbers to process ([0-2] for test, [0-16] for train)
    - data_path_input: input directory containing bucket folders
    - data_path_output_event: output directory for event files
    - data_path_output_noise: output directory for noise files
    - metadata: pd dataframe with trace metadata (including SNR values)
    - sampling_rate: sampling rate in Hz (default 100000 Hz)
    - snr_threshold: minimum SNR threshold to save traces (default 200)
    """
    
    # Create output directories
    os.makedirs(data_path_output_event, exist_ok=True) #they already exist, but just in case
    os.makedirs(data_path_output_noise, exist_ok=True)
    
    print(f"Processing {buckets} buckets...")
    print(f"Input directory: {data_path_input}")
    print(f"Event output: {data_path_output_event}")
    print(f"Noise output: {data_path_output_noise}")
    print(f"SNR threshold: {snr_threshold}")
    
    # Sample indices for resampled data
    noise_start = int(0.005 * sampling_rate)  #  # 500 samples at 100kHz
    noise_end_event_start = int(0.07 * sampling_rate) # 7000 samples at 100kHz
    event_end = int(0.15 * sampling_rate) # 15000 samples at 100kHz
    
    event_counter = 0
    noise_counter = 0
    skipped_counter = 0
    
    for bucket_num in range(buckets):
        input_path = os.path.join(data_path_input, f'bucket_{bucket_num:02d}_highpass.npz') # individual bucket files
        
        if os.path.exists(input_path):
            print(f"Processing bucket {bucket_num}...")
            
            # Load bucket data
            data = np.load(input_path)
            waveforms = data['waveforms']
            num_traces = data['num_traces']
            
            print(f"  Found {num_traces} traces in bucket {bucket_num}")
            
            # Process each trace in the bucket
            for trace_idx in range(num_traces):
                # variation in trace naming format
                # Try all possible trace_name variants with x in [29990, 30010]
                found = False
                for x in range(29990, 30011):
                    trace_name_candidate = f'bucket{bucket_num}${trace_idx},:1,:{x}'
                    mask = metadata['trace_name'] == trace_name_candidate
                    if mask.any():
                        SNR = metadata.loc[mask, 'trace_snr'].values[0]
                        found = True
                        break
                if not found:
                    print(f"  Warning: Could not find metadata for bucket{bucket_num}${trace_idx} in range 29990-30010")
                    skipped_counter += 1
                    continue
                
                # Skip traces below SNR threshold
                if SNR < snr_threshold:
                    skipped_counter += 1
                    continue
                
                trace_data = waveforms[trace_idx, 0, :]  # Single component [trace, component=0, samples]
                
                # Extract noise (0.005-0.07s)
                trace_data_noise = trace_data[noise_start:noise_end_event_start]
                
                # Extract event (0.07-0.15s) 
                trace_data_event = trace_data[noise_end_event_start:event_end]
                
                # Save noise file
                noise_filename = f'Noise{noise_counter:06d}.npz'
                noise_filepath = os.path.join(data_path_output_noise, noise_filename)
                np.savez_compressed(noise_filepath,
                                  noise_waveform_Z=trace_data_noise,  # Keep naming convention in ModelTrainingWaveforms .py
                                  bucket_number=bucket_num,
                                  trace_index=trace_idx,
                                  snr=SNR,
                                  sampling_rate=sampling_rate)
                
                # Save event file
                event_filename = f'Event{event_counter:06d}.npz'
                event_filepath = os.path.join(data_path_output_event, event_filename)
                np.savez_compressed(event_filepath,
                                  processed_waveform_Z=trace_data_event,  # Keep same naming convention as in ModelTrainingWaveforms .py
                                  bucket_number=bucket_num,
                                  trace_index=trace_idx,
                                  snr=SNR,
                                  sampling_rate=sampling_rate)
                
                noise_counter += 1
                event_counter += 1
                
                # Progress indicator
                if (trace_idx + 1) % 100 == 0:
                    print(f"    Processed {trace_idx + 1}/{num_traces} traces")
            
            print(f"  Completed bucket {bucket_num}: {num_traces} traces processed")
            
        else:
            print(f"Warning: {input_path} not found")
    
    print(f"\nConversion complete!")
    print(f"Created {event_counter} event files in: {data_path_output_event}")
    print(f"Created {noise_counter} noise files in: {data_path_output_noise}")
    print(f"Skipped {skipped_counter} traces (SNR < {snr_threshold} or missing metadata)")
    print(f"Acceptance rate: {event_counter/(event_counter + skipped_counter)*100:.1f}%")
    
    return event_counter, noise_counter

In [ ]:
event, noise = buckets_to_event_noise_files(
    buckets=13,  
    data_path_input='npz_files_100kHz_highpass_bworder4_zerophase',
    data_path_output_event='single_waveforms/event',
    data_path_output_noise='single_waveforms/noise',
    metadata=metadata,
    sampling_rate=100000,
    snr_threshold=3
)

In [ ]:
noise_dir = 'single_waveforms/noise'
noise_files = glob.glob(os.path.join(noise_dir, 'Noise*.npz'))

plt.figure(figsize=(12, 6))
for i in range(100):
    file = np.random.choice(noise_files)
    data = np.load(file)
    waveform = data['noise_waveform_Z']
    time_axis = np.arange(len(waveform)) / 100000  # Assuming 100kHz sampling rate
    plt.plot(time_axis, waveform, alpha=0.7, label=f'Noise {i+1}' if i == 0 else "")
plt.title('20 Random Noise Waveforms')
plt.xlabel('Sample Index')
plt.ylabel('Amplitude')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
def analyze_noise_traces_time_domain(noise_dir, num_samples=None, plot_median=True, check_trends=True):
    """
    Analyze noise traces in time domain and check for statistical trends
    
    Parameters:
    - noise_dir: Directory containing noise .npz files
    - num_samples: Number of noise files to analyze (None = all files)
    - plot_median: Whether to plot median trace
    - check_trends: Whether to perform statistical trend analysis
    """
    import matplotlib.pyplot as plt
    from scipy import stats
    import glob
    
    # Get all noise files
    noise_files = glob.glob(os.path.join(noise_dir, 'Noise*.npz'))
    
    if num_samples is not None:
        noise_files = noise_files[:num_samples]
    
    print(f"Analyzing {len(noise_files)} noise traces...")
    
    # Load all noise traces
    all_noise_traces = []
    snr_values = []
    
    for i, noise_file in enumerate(noise_files):
        data = np.load(noise_file)
        noise_trace = data['noise_waveform_Z']
        snr = data['snr']
        
        all_noise_traces.append(noise_trace)
        snr_values.append(snr)
        
        if (i + 1) % 1000 == 0:
            print(f"  Loaded {i + 1}/{len(noise_files)} traces")
    
    # Convert to numpy array
    all_noise_traces = np.array(all_noise_traces)  # Shape: (n_traces, 4000)
    snr_values = np.array(snr_values)
    
    print(f"Loaded {all_noise_traces.shape[0]} traces, each with {all_noise_traces.shape[1]} samples")
    
    # Calculate statistics
    median_trace = np.median(all_noise_traces, axis=0)
    mean_trace = np.mean(all_noise_traces, axis=0)
    std_trace = np.std(all_noise_traces, axis=0)
    percentile_25 = np.percentile(all_noise_traces, 25, axis=0)
    percentile_75 = np.percentile(all_noise_traces, 75, axis=0)
    percentile_5 = np.percentile(all_noise_traces, 5, axis=0)
    percentile_95 = np.percentile(all_noise_traces, 95, axis=0)
    
    # Time axis (0.07 seconds at 100kHz)
    sampling_rate = 100000
    time_axis = np.linspace(0.005, 0.07, all_noise_traces.shape[1])
    
    if plot_median:
        # Create time domain plot
        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10))
        
        # Plot 1: Median trace with percentile bands
        ax1.fill_between(time_axis, percentile_5, percentile_95, alpha=0.2, color='blue', label='90th percentile band')
        ax1.fill_between(time_axis, percentile_25, percentile_75, alpha=0.4, color='blue', label='50th percentile band')
        ax1.plot(time_axis, median_trace, 'b-', linewidth=2, label='Median trace')
        ax1.plot(time_axis, mean_trace, 'r--', linewidth=1, label='Mean trace')
        ax1.set_xlabel('Time (s)')
        ax1.set_ylabel('Amplitude')
        ax1.set_title(f'Noise Traces Time Domain Analysis (train dataset, {len(noise_files)} traces)')
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # Plot 2: Standard deviation over time
        ax2.plot(time_axis, std_trace, 'g-', linewidth=2, label='Standard deviation')
        ax2.set_xlabel('Time (s)')
        ax2.set_ylabel('Standard Deviation')
        ax2.set_title('Variability of Noise Traces Over Time')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
        # Plot 3: Example individual traces
        ax3.plot(time_axis, all_noise_traces[0], 'gray', alpha=0.5, label='Individual traces')
        for i in range(1, min(10, len(all_noise_traces))):
            ax3.plot(time_axis, all_noise_traces[i], 'gray', alpha=0.5)
        ax3.plot(time_axis, median_trace, 'b-', linewidth=2, label='Median trace')
        ax3.set_xlabel('Time (s)')
        ax3.set_ylabel('Amplitude')
        ax3.set_title('Individual Noise Traces vs Median')
        ax3.grid(True, alpha=0.3)
        ax3.legend()
        
        plt.tight_layout()
        plt.show()
    
    if check_trends:
        print("\n" + "="*50)
        print("STATISTICAL TREND ANALYSIS")
        print("="*50)
        
        # 1. Overall amplitude statistics
        trace_rms = np.sqrt(np.mean(all_noise_traces**2, axis=1))
        trace_max = np.max(np.abs(all_noise_traces), axis=1)
        trace_std = np.std(all_noise_traces, axis=1)
        
        print(f"\nOverall Amplitude Statistics:")
        print(f"RMS values - Mean: {np.mean(trace_rms):.2e}, Std: {np.std(trace_rms):.2e}")
        print(f"Max values - Mean: {np.mean(trace_max):.2e}, Std: {np.std(trace_max):.2e}")
        print(f"Std values - Mean: {np.mean(trace_std):.2e}, Std: {np.std(trace_std):.2e}")
        
        # 2. Check for correlation with SNR
        corr_rms_snr, p_rms_snr = stats.pearsonr(trace_rms, snr_values)
        corr_max_snr, p_max_snr = stats.pearsonr(trace_max, snr_values)
        
        print(f"\nCorrelation with SNR:")
        print(f"RMS vs SNR: r = {corr_rms_snr:.3f}, p = {p_rms_snr:.3e}")
        print(f"Max vs SNR: r = {corr_max_snr:.3f}, p = {p_max_snr:.3e}")
        
        # 3. Check for temporal trends (drift over file index)
        file_indices = np.arange(len(noise_files))
        corr_rms_time, p_rms_time = stats.pearsonr(trace_rms, file_indices)
        corr_max_time, p_max_time = stats.pearsonr(trace_max, file_indices)
        
        print(f"\nTemporal Trends (over file order):")
        print(f"RMS vs file index: r = {corr_rms_time:.3f}, p = {p_rms_time:.3e}")
        print(f"Max vs file index: r = {corr_max_time:.3f}, p = {p_max_time:.3e}")
        
        # 4. Check for normality
        _, p_rms_normal = stats.shapiro(trace_rms[:1000] if len(trace_rms) > 1000 else trace_rms)
        _, p_max_normal = stats.shapiro(trace_max[:1000] if len(trace_max) > 1000 else trace_max)
        
        print(f"\nNormality Tests (Shapiro-Wilk):")
        print(f"RMS normality: p = {p_rms_normal:.3e} ({'Normal' if p_rms_normal > 0.05 else 'Non-normal'})")
        print(f"Max normality: p = {p_max_normal:.3e} ({'Normal' if p_max_normal > 0.05 else 'Non-normal'})")
        
        # 5. Distribution plots
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 8))
        
        # RMS distribution
        ax1.hist(trace_rms, bins=50, alpha=0.7, color='blue', edgecolor='black')
        ax1.set_xlabel('RMS Amplitude')
        ax1.set_ylabel('Count')
        ax1.set_title('Distribution of RMS Values')
        ax1.grid(True, alpha=0.3)
        
        # Max amplitude distribution
        ax2.hist(trace_max, bins=50, alpha=0.7, color='red', edgecolor='black')
        ax2.set_xlabel('Max Amplitude')
        ax2.set_ylabel('Count')
        ax2.set_title('Distribution of Max Values')
        ax2.grid(True, alpha=0.3)
        
        # RMS vs SNR scatter
        ax3.scatter(snr_values, trace_rms, alpha=0.5, s=1)
        ax3.set_xlabel('SNR')
        ax3.set_ylabel('RMS Amplitude')
        ax3.set_title(f'RMS vs SNR (r={corr_rms_snr:.3f})')
        ax3.grid(True, alpha=0.3)
        
        # RMS vs file index (temporal trend)
        ax4.scatter(file_indices, trace_rms, alpha=0.5, s=1)
        ax4.set_xlabel('File Index')
        ax4.set_ylabel('RMS Amplitude')
        ax4.set_title(f'RMS vs File Order (r={corr_rms_time:.3f})')
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # 6. Summary assessment
        print(f"\n" + "="*50)
        print("TREND SUMMARY")
        print("="*50)
        
        significant_correlations = []
        if abs(corr_rms_snr) > 0.1 and p_rms_snr < 0.05:
            significant_correlations.append(f"RMS-SNR correlation: {corr_rms_snr:.3f}")
        if abs(corr_rms_time) > 0.1 and p_rms_time < 0.05:
            significant_correlations.append(f"RMS-temporal trend: {corr_rms_time:.3f}")
        if abs(corr_max_snr) > 0.1 and p_max_snr < 0.05:
            significant_correlations.append(f"Max-SNR correlation: {corr_max_snr:.3f}")
        if abs(corr_max_time) > 0.1 and p_max_time < 0.05:
            significant_correlations.append(f"Max-temporal trend: {corr_max_time:.3f}")
        
        if significant_correlations:
            print("Significant trends detected:")
            for corr in significant_correlations:
                print(f"  - {corr}")
        else:
            print("No significant trends detected - noise appears stationary")
    
    return {
        'median_trace': median_trace,
        'mean_trace': mean_trace,
        'std_trace': std_trace,
        'all_traces': all_noise_traces,
        'snr_values': snr_values,
        'time_axis': time_axis,
        'statistics': {
            'rms_values': trace_rms if check_trends else None,
            'max_values': trace_max if check_trends else None,
            'correlations': {
                'rms_snr': (corr_rms_snr, p_rms_snr) if check_trends else None,
                'rms_temporal': (corr_rms_time, p_rms_time) if check_trends else None
            }
        }
    }

# Run the analysis
results = analyze_noise_traces_time_domain('single_waveforms/noise')

In [ ]:
def analyze_event_traces_time_domain(event_dir, num_samples=None, plot_median=True, check_trends=True):
    """
    Analyze event traces in time domain and check for statistical trends
    
    Parameters:
    - event_dir: Directory containing event .npz files
    - num_samples: Number of event files to analyze (None = all files)
    - plot_median: Whether to plot median trace
    - check_trends: Whether to perform statistical trend analysis
    """
    import matplotlib.pyplot as plt
    from scipy import stats
    import glob
    
    # Get all event files
    event_files = glob.glob(os.path.join(event_dir, 'Event*.npz'))
    
    if num_samples is not None:
        event_files = event_files[:num_samples]
    
    print(f"Analyzing {len(event_files)} event traces...")
    
    # Load all event traces
    all_event_traces = []
    snr_values = []
    
    for i, event_file in enumerate(event_files):
        data = np.load(event_file)
        event_trace = data['processed_waveform_Z']  # Changed from noise_waveform_Z
        snr = data['snr']
        
        all_event_traces.append(event_trace)
        snr_values.append(snr)
        
        if (i + 1) % 1000 == 0:
            print(f"  Loaded {i + 1}/{len(event_files)} traces")
    
    # Convert to numpy array
    all_event_traces = np.array(all_event_traces)  # Shape: (n_traces, 4000)
    snr_values = np.array(snr_values)
    
    print(f"Loaded {all_event_traces.shape[0]} traces, each with {all_event_traces.shape[1]} samples")
    
    # Calculate statistics
    median_trace = np.median(all_event_traces, axis=0)
    mean_trace = np.mean(all_event_traces, axis=0)
    std_trace = np.std(all_event_traces, axis=0)
    percentile_25 = np.percentile(all_event_traces, 25, axis=0)
    percentile_75 = np.percentile(all_event_traces, 75, axis=0)
    percentile_5 = np.percentile(all_event_traces, 5, axis=0)
    percentile_95 = np.percentile(all_event_traces, 95, axis=0)
    
    # Time axis (0.08 seconds at 100kHz)
    sampling_rate = 100000
    time_axis = np.linspace(0, 0.08, all_event_traces.shape[1])
    
    if plot_median:
        # Create time domain plot
        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10))
        
        # Plot 1: Median trace with percentile bands
        ax1.fill_between(time_axis, percentile_5, percentile_95, alpha=0.2, color='red', label='90th percentile band')
        ax1.fill_between(time_axis, percentile_25, percentile_75, alpha=0.4, color='red', label='50th percentile band')
        ax1.plot(time_axis, median_trace, 'r-', linewidth=2, label='Median trace')
        ax1.plot(time_axis, mean_trace, 'b--', linewidth=1, label='Mean trace')
        ax1.set_xlabel('Time (s)')
        ax1.set_ylabel('Amplitude')
        ax1.set_title(f'Event Traces Time Domain Analysis (train dataset, {len(event_files)} traces)')
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # Plot 2: Standard deviation over time
        ax2.plot(time_axis, std_trace, 'g-', linewidth=2, label='Standard deviation')
        ax2.set_xlabel('Time (s)')
        ax2.set_ylabel('Standard Deviation')
        ax2.set_title('Variability of Event Traces Over Time')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
        # Plot 3: Example individual traces
        ax3.plot(time_axis, all_event_traces[0], 'gray', alpha=0.5, label='Individual traces')
        for i in range(1, min(10, len(all_event_traces))):
            ax3.plot(time_axis, all_event_traces[i], 'gray', alpha=0.5)
        ax3.plot(time_axis, median_trace, 'r-', linewidth=2, label='Median trace')
        ax3.set_xlabel('Time (s)')
        ax3.set_ylabel('Amplitude')
        ax3.set_title('Individual Event Traces vs Median')
        ax3.grid(True, alpha=0.3)
        ax3.legend()
        
        plt.tight_layout()
        plt.show()
    
    if check_trends:
        print("\n" + "="*50)
        print("STATISTICAL TREND ANALYSIS")
        print("="*50)
        
        # 1. Overall amplitude statistics
        trace_rms = np.sqrt(np.mean(all_event_traces**2, axis=1))
        trace_max = np.max(np.abs(all_event_traces), axis=1)
        trace_std = np.std(all_event_traces, axis=1)
        
        print(f"\nOverall Amplitude Statistics:")
        print(f"RMS values - Mean: {np.mean(trace_rms):.2e}, Std: {np.std(trace_rms):.2e}")
        print(f"Max values - Mean: {np.mean(trace_max):.2e}, Std: {np.std(trace_max):.2e}")
        print(f"Std values - Mean: {np.mean(trace_std):.2e}, Std: {np.std(trace_std):.2e}")
        
        # 2. Check for correlation with SNR
        corr_rms_snr, p_rms_snr = stats.pearsonr(trace_rms, snr_values)
        corr_max_snr, p_max_snr = stats.pearsonr(trace_max, snr_values)
        
        print(f"\nCorrelation with SNR:")
        print(f"RMS vs SNR: r = {corr_rms_snr:.3f}, p = {p_rms_snr:.3e}")
        print(f"Max vs SNR: r = {corr_max_snr:.3f}, p = {p_max_snr:.3e}")
        
        # 3. Check for temporal trends (drift over file index)
        file_indices = np.arange(len(event_files))
        corr_rms_time, p_rms_time = stats.pearsonr(trace_rms, file_indices)
        corr_max_time, p_max_time = stats.pearsonr(trace_max, file_indices)
        
        print(f"\nTemporal Trends (over file order):")
        print(f"RMS vs file index: r = {corr_rms_time:.3f}, p = {p_rms_time:.3e}")
        print(f"Max vs file index: r = {corr_max_time:.3f}, p = {p_max_time:.3e}")
        
        # 4. Check for normality
        _, p_rms_normal = stats.shapiro(trace_rms[:1000] if len(trace_rms) > 1000 else trace_rms)
        _, p_max_normal = stats.shapiro(trace_max[:1000] if len(trace_max) > 1000 else trace_max)
        
        print(f"\nNormality Tests (Shapiro-Wilk):")
        print(f"RMS normality: p = {p_rms_normal:.3e} ({'Normal' if p_rms_normal > 0.05 else 'Non-normal'})")
        print(f"Max normality: p = {p_max_normal:.3e} ({'Normal' if p_max_normal > 0.05 else 'Non-normal'})")
        
        # 5. Distribution plots
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 8))
        
        # RMS distribution
        ax1.hist(trace_rms, bins=50, alpha=0.7, color='red', edgecolor='black')
        ax1.set_xlabel('RMS Amplitude')
        ax1.set_ylabel('Count')
        ax1.set_title('Distribution of RMS Values')
        ax1.grid(True, alpha=0.3)
        
        # Max amplitude distribution
        ax2.hist(trace_max, bins=50, alpha=0.7, color='orange', edgecolor='black')
        ax2.set_xlabel('Max Amplitude')
        ax2.set_ylabel('Count')
        ax2.set_title('Distribution of Max Values')
        ax2.grid(True, alpha=0.3)
        
        # RMS vs SNR scatter
        ax3.scatter(snr_values, trace_rms, alpha=0.5, s=1)
        ax3.set_xlabel('SNR')
        ax3.set_ylabel('RMS Amplitude')
        ax3.set_title(f'RMS vs SNR (r={corr_rms_snr:.3f})')
        ax3.grid(True, alpha=0.3)
        
        # RMS vs file index (temporal trend)
        ax4.scatter(file_indices, trace_rms, alpha=0.5, s=1)
        ax4.set_xlabel('File Index')
        ax4.set_ylabel('RMS Amplitude')
        ax4.set_title(f'RMS vs File Order (r={corr_rms_time:.3f})')
        ax4.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # 6. Summary assessment
        print(f"\n" + "="*50)
        print("TREND SUMMARY")
        print("="*50)
        
        significant_correlations = []
        if abs(corr_rms_snr) > 0.1 and p_rms_snr < 0.05:
            significant_correlations.append(f"RMS-SNR correlation: {corr_rms_snr:.3f}")
        if abs(corr_rms_time) > 0.1 and p_rms_time < 0.05:
            significant_correlations.append(f"RMS-temporal trend: {corr_rms_time:.3f}")
        if abs(corr_max_snr) > 0.1 and p_max_snr < 0.05:
            significant_correlations.append(f"Max-SNR correlation: {corr_max_snr:.3f}")
        if abs(corr_max_time) > 0.1 and p_max_time < 0.05:
            significant_correlations.append(f"Max-temporal trend: {corr_max_time:.3f}")
        
        if significant_correlations:
            print("Significant trends detected:")
            for corr in significant_correlations:
                print(f"  - {corr}")
        else:
            print("No significant trends detected - events appear stationary")
    
    return {
        'median_trace': median_trace,
        'mean_trace': mean_trace,
        'std_trace': std_trace,
        'all_traces': all_event_traces,
        'snr_values': snr_values,
        'time_axis': time_axis,
        'statistics': {
            'rms_values': trace_rms if check_trends else None,
            'max_values': trace_max if check_trends else None,
            'correlations': {
                'rms_snr': (corr_rms_snr, p_rms_snr) if check_trends else None,
                'rms_temporal': (corr_rms_time, p_rms_time) if check_trends else None
            }
        }
    }

# Run the analysis for events
event_results = analyze_event_traces_time_domain('single_waveforms/event')

In [ ]:
def analyze_frequency_content_multiple_traces(resampled_dir, metadata, SNR_highpass=True, SNR_highpass_freq=1000):
    """
    Analyze frequency content across all resampled buckets 
    params:
    - resampled_dir: Directory containing resampled .npz files
    - metadata: Metadata DataFrame
    """
    import matplotlib.pyplot as plt
    
    sampling_rate = 100000  # Resampled rate
    
    # Collect amplitude data from all buckets
    all_amplitudes_noise = []
    frequencies_noise = None
    all_amplitudes_event = []
    frequencies_event = None
    
    # Process all 17 buckets 
    for bucket_num in range(17):
        npz_path = os.path.join(resampled_dir, f'bucket_{bucket_num:02d}_highpass.npz')
        
        if os.path.exists(npz_path):
            # Load data for this specific bucket
            data = np.load(npz_path)
            bucket_data = data['waveforms']
            num_traces = bucket_data.shape[0]  # Get number of traces for this bucket
            
            for i in range(num_traces):
                trace_data = bucket_data[i, 0, :]
                
                if SNR_highpass:
                    # Try both possible trace name formats
                    location_20001 = f'bucket{bucket_num}${i},:1,:20001'
                    location_20000 = f'bucket{bucket_num}${i},:1,:20000'
                    
                    # Check which one exists in metadata
                    mask_20001 = metadata['trace_name'] == location_20001
                    mask_20000 = metadata['trace_name'] == location_20000
                    
                    if mask_20001.any():
                        SNR = metadata.loc[mask_20001, 'trace_snr'].values[0]
                    elif mask_20000.any():
                        SNR = metadata.loc[mask_20000, 'trace_snr'].values[0]
                    else:
                        print(f"Warning: Could not find metadata for bucket{bucket_num}${i}")
                        continue
                        
                    if SNR < SNR_highpass_freq:
                        continue
                
                # Calculate sample indices for resampled data (100kHz)
                noise_samples = int(0.05 * sampling_rate)  # 5000 samples at 100kHz
                event_samples = int(0.05 * sampling_rate)  # 5000 samples at 100kHz
                
                trace_data_noise = trace_data[:noise_samples]
                trace_data_event = trace_data[noise_samples:noise_samples + event_samples]
                
                # FFT analysis with windowing
                windowed_data_event = trace_data_event * np.hanning(len(trace_data_event))
                windowed_data_noise = trace_data_noise * np.hanning(len(trace_data_noise))

                fft_result_event = np.fft.fft(windowed_data_event)
                fft_result_noise = np.fft.fft(windowed_data_noise)
                freq_temp_event = np.fft.fftfreq(len(windowed_data_event), 1/sampling_rate)
                freq_temp_noise = np.fft.fftfreq(len(windowed_data_noise), 1/sampling_rate)
                
                # Get positive frequencies
                positive_freq_mask = freq_temp_event > 0
                freq_positive_event = freq_temp_event[positive_freq_mask]
                amplitude_positive_event = np.abs(fft_result_event[positive_freq_mask])

                positive_freq_mask = freq_temp_noise > 0
                freq_positive_noise = freq_temp_noise[positive_freq_mask]
                amplitude_positive_noise = np.abs(fft_result_noise[positive_freq_mask])

                # Store frequencies reference
                if frequencies_event is None:
                    frequencies_event = freq_positive_event
                all_amplitudes_event.append(amplitude_positive_event)
                
                if frequencies_noise is None:
                    frequencies_noise = freq_positive_noise
                all_amplitudes_noise.append(amplitude_positive_noise)
                
            print(f"Processed bucket {bucket_num}: {num_traces} traces (total {len(all_amplitudes_event)} traces)")
        else:
            print(f"Warning: {npz_path} not found")
    
    # Convert to numpy array for statistics
    all_amplitudes_event = np.array(all_amplitudes_event)
    all_amplitudes_noise = np.array(all_amplitudes_noise)
    
    # Calculate percentiles for interval event
    median_amp_event = np.median(all_amplitudes_event, axis=0)
    percentile_25_event = np.percentile(all_amplitudes_event, 25, axis=0)
    percentile_75_event = np.percentile(all_amplitudes_event, 75, axis=0)
    percentile_5_event = np.percentile(all_amplitudes_event, 5, axis=0)
    percentile_95_event = np.percentile(all_amplitudes_event, 95, axis=0)
    
    # Calculate percentiles for interval noise
    median_amp_noise = np.median(all_amplitudes_noise, axis=0)
    percentile_25_noise = np.percentile(all_amplitudes_noise, 25, axis=0)
    percentile_75_noise = np.percentile(all_amplitudes_noise, 75, axis=0)
    percentile_5_noise = np.percentile(all_amplitudes_noise, 5, axis=0)
    percentile_95_noise = np.percentile(all_amplitudes_noise, 95, axis=0)

    # Create interval plot
    fig, ax1 = plt.subplots(1, 1, figsize=(12, 8))
    
    # Plot intervals
    ax1.fill_between(frequencies_event, percentile_5_event, percentile_95_event, alpha=0.2, color='red', label='90th percentile band - event')
    ax1.fill_between(frequencies_event, percentile_25_event, percentile_75_event, alpha=0.4, color='red', label='50th percentile band - event')
    ax1.loglog(frequencies_event, median_amp_event, 'r-', linewidth=2, label='Median amplitude event')

    ax1.fill_between(frequencies_noise, percentile_5_noise, percentile_95_noise, alpha=0.2, color='blue', label='90th percentile band - noise')
    ax1.fill_between(frequencies_noise, percentile_25_noise, percentile_75_noise, alpha=0.4, color='blue', label='50th percentile band - noise')
    ax1.loglog(frequencies_noise, median_amp_noise, 'b-', linewidth=2, label='Median amplitude noise')
    ax1.set_ylim(10, 1e6)  # dont focus on low amplitude high frequency part of the plot.
    # Add Nyquist frequency line
    nyquist_freq = sampling_rate / 2
    ax1.axvline(x=nyquist_freq, color='red', linestyle='--', linewidth=2, 
               label=f'Nyquist frequency ({nyquist_freq:.0f} Hz)')
    
    ax1.set_xlabel('Frequency (Hz)')
    ax1.set_ylabel('Amplitude')
    ax1.set_title(f'SNR above {SNR_highpass_freq} ({all_amplitudes_event.shape[0]} traces)')
    ax1.grid(True, alpha=0.3, which='both')
    ax1.legend()
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nAnalyzed {all_amplitudes_event.shape[0]} traces from all buckets")
    print(f"Sampling rate: {sampling_rate} Hz")
    print(f"Nyquist frequency: {nyquist_freq} Hz")

# Run with corrected function call
analyze_frequency_content_multiple_traces('npz_files_100kHz_highpass', metadata)